### probar modelo fusionando clases

In [1]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado/"

# Rutas de TuckER 5D Balanceado
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"
    r"\best_model.pt"
)

# Rutas de Predictores 5D
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_standard"
    r"\best_predictor_dim5_rdim{rdim}_standard_5d.pt"
)

# Datos
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")
# Archivo de Puntajes
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones ORIGINALES del modelo (las que rankeamos)
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17)

DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES
# ============================================================

sys.path.append(r"C:\Users\56946\TuckER")
from load_data import Data

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe: {predictor_path}")
    state = torch.load(predictor_path, map_location=device)
    model.load_state_dict(pick_state_dict(state))
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe: {tucker_path}")
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def build_vocab(data_dir, reverse=True):
    d = Data(data_dir=data_dir, reverse=reverse)
    entities, relations = d.entities, d.relations
    ent2idx = {e:i for i,e in enumerate(entities)}
    rel2idx = {r:i for i,r in enumerate(relations)}
    return SimpleNamespace(entities=entities, relations=relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ⚠️ VECTOR 5D (Notas + Puntaje)
def get_notes_vector_5d(csv_path, puntajes_map, max_score, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, 5), dtype=torch.float32) # Dim 5
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    vec_notas = np.zeros(4, dtype=np.float32)
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            vec_notas[idx_map[c]] = float(row["NOTA"]) / 7.0
            
    # Puntaje
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val): puntaje_norm = -1.0
    else: puntaje_norm = score_val / max_score
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 CARGA Y PREPARACIÓN DE DATOS
# ============================================================

print("Cargando Dataframes...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

# Cargar Puntajes
print("Cargando Puntajes...")
if os.path.exists(RUTA_PUNTAJES):
    df_puntajes = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    # Max Score Global
    max_score_global = df_puntajes["PUNTAJE_PONDERADO"].max()
    if pd.isna(max_score_global): max_score_global = 850.0
    puntajes_map = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
else:
    print("⚠️ No se encontró archivo de puntajes. Usando -1 por defecto.")
    puntajes_map = {}
    max_score_global = 850.0

for df in (df_20211, df_20212):
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()
print(f"🔹 {len(alumnos_validos)} alumnos válidos 20211.")

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

# Definimos la categoría real (sin fusionar todavía, para tener el detalle si se necesita)
def categoria_relacion(nota):
    if pd.isna(nota) or nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval["NOTA"].apply(categoria_relacion)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 LOOP EVALUACIÓN
# ============================================================

resumen_metricas = []

for rdim in RDIMS:
    print(f"\n============================== Evaluando rdim={rdim} (5D) ==============================")

    tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
    predictor_path = PRED_DIR_FMT.format(rdim=rdim)

    if not os.path.exists(tucker_path):
        print(f"⚠️ Faltan archivos TuckER")
        continue

    d = build_vocab(DATA_DIR, reverse=True)
    E, R, W = load_tucker_weights(tucker_path, DEVICE)
    d1 = E.shape[1]

    rel2idx_map = {}
    for rel_hint in RELACIONES:
        idx = find_relation(d.relations, d.relation_idxs, hint=rel_hint)
        if idx is not None: rel2idx_map[rel_hint] = idx

    try:
        # ⚠️ Input Size = 5
        predictor = load_predictor(predictor_path, input_size=5, out_dim=d1, device="cpu")
    except Exception as e:
        print(f"⚠️ Error cargando predictor: {e}")
        continue

    ehat_by_alumno = {}
    for aid in df_eval["ID"].unique():
        # Usar función 5D
        x = get_notes_vector_5d(CSV_20211, puntajes_map, max_score_global, aid, CURSOS_PRIMER)
        with torch.no_grad(): ehat_by_alumno[aid] = predictor(x).squeeze(0)

    # Listas para métricas fusionadas
    y_true_fusionado = []
    y_pred_fusionado = []

    for _, row in df_eval.iterrows():
        aid, curso, true_rel = row["ID"], row["CURSO"], row["RELACION_REAL"]
        if curso not in d.entity_idxs: continue
        
        t_idx = d.entity_idxs[curso]
        e_hat = ehat_by_alumno[aid]

        scores = {}
        for r_name, r_idx in rel2idx_map.items():
            scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
        
        # Predicción original (multiclase)
        pred_rel = max(scores, key=scores.get)
        
        # --- FUSIÓN DE CLASES ---
        # Si contiene 'reprueba' es Reprueba (1).
        # Si contiene 'aprueba' (en cualquiera de sus formas) es Aprueba (0).
        # Usamos 1 para Reprueba para enfocar las métricas (Recall/Precision) en la clase minoritaria de interés.
        
        real_bin = 1 if "reprueba" in true_rel.lower() else 0
        pred_bin = 1 if "reprueba" in pred_rel.lower() else 0
        
        y_true_fusionado.append(real_bin)
        y_pred_fusionado.append(pred_bin)

    if not y_true_fusionado: continue

    # Métricas Binarias (Con fusión implícita)
    # TN: Real=Aprueba, Pred=Aprueba
    # FP: Real=Aprueba, Pred=Reprueba (Falsa Alarma)
    # FN: Real=Reprueba, Pred=Aprueba (No detectado)
    # TP: Real=Reprueba, Pred=Reprueba (Detectado)
    tn, fp, fn, tp = confusion_matrix(y_true_fusionado, y_pred_fusionado).ravel()
    
    total_reales_reprobados = tp + fn
    total_predichos_reprobados = tp + fp
    
    recall = tp / total_reales_reprobados if total_reales_reprobados > 0 else 0
    precision = tp / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
    accuracy = (tp + tn) / len(y_true_fusionado)
    
    print(f"📊 Resultados Binarios Fusionados (Focus: Reprobación)")
    print("-" * 60)
    print(f"Accuracy Global         : {accuracy:.4f}")
    print(f"✅ RECALL (Sensibilidad) : {recall:.4f} ({tp}/{total_reales_reprobados})")
    print(f"🎯 PRECISION             : {precision:.4f}")
    print(f"Matriz: [TP={tp} FN={fn}] | [FP={fp} TN={tn}]")

Cargando Dataframes...
Cargando Puntajes...
🔹 830 alumnos válidos 20211.
Evaluaciones totales: 3023

============================== Evaluando rdim=1 (5D) ==============================
📊 Resultados Binarios Fusionados (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.9229
✅ RECALL (Sensibilidad) : 0.0000 (0/233)
🎯 PRECISION             : 0.0000
Matriz: [TP=0 FN=233] | [FP=0 TN=2790]

============================== Evaluando rdim=2 (5D) ==============================
📊 Resultados Binarios Fusionados (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.0771
✅ RECALL (Sensibilidad) : 1.0000 (233/233)
🎯 PRECISION             : 0.0771
Matriz: [TP=233 FN=0] | [FP=2790 TN=0]

============================== Evaluando rdim=3 (5D) ==============================
📊 Resultados Binarios Fusionados (Focus: Reprobación)
------------------------------------------------------------
Accura

# Generar redes neuronales balanceadas

In [1]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# CONFIGURACIÓN GENERAL
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# --- RUTAS DE ENTRADA ---
# Dataset TuckER (Multinivel)
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado"
RESULTS_BASE    = r"C:\Users\56946\TuckER\results"

# Prefijo del TuckER 5D Balanceado (Coincide con tu último .bat)
RUN_PREFIX      = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"

# ⚠️ NUEVA RUTA DE SALIDA SOLICITADA
SAVE_BASE       = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d"

# --- DATOS ---
RUTA_DF_INPUT  = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\df_20191.csv" # Historia
RUTA_DF_TARGET = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\df_20192.csv" # Para balanceo

# Archivo de Puntajes
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER  = ['MA1101', 'MA1001', 'FI1000', 'BT1211']
# Para identificar reprobados miramos 1ro y 2do sem
CURSOS_EVAL    = CURSOS_PRIMER + ['MA1002', 'MA1102', 'FI1100', 'CC1002'] 

RDIMS = range(1, 17)

# =========================
# FUNCIONES AUXILIARES
# =========================
def get_vocab_from_data_dir(data_dir):
    entities = set()
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    h, r, t = line.strip().split()
                    entities.add(h.strip().upper())
    return sorted(list(entities))

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def cargar_E_weights(path_pt):
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    return sd["E.weight"].detach().cpu()

def limpiar_nota(nota_str, estado):
    if isinstance(estado, str) and "Reprobado" in estado: return 1.0
    try:
        if pd.isna(nota_str) or str(nota_str).strip() == "": return 0.0
        return float(str(nota_str).replace(",", "."))
    except: return 0.0

# ⚠️ VECTOR 5D (4 Notas + 1 Puntaje)
def preparar_X_historico_5dim(ruta_input, ruta_puntajes, cursos_primer):
    print(f"🔄 Construyendo vectores históricos de dimensión 5 (4 Notas + Puntaje)...")
    
    # 1. Cargar Notas
    df = pd.read_csv(ruta_input, sep=';')
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    todos_alumnos = df["ID"].unique()
    
    # 2. Cargar y Normalizar Puntajes
    df_puntajes = pd.read_csv(ruta_puntajes, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    # Máximo local para normalizar
    max_score = df_puntajes[df_puntajes["ID"].isin(todos_alumnos)]["PUNTAJE_PONDERADO"].max()
    if pd.isna(max_score): max_score = 850.0
    print(f"   Max Puntaje detectado para normalizar: {max_score}")

    mapa_puntajes = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()

    idx_primer = {c: i for i, c in enumerate(cursos_primer)}
    vectores = {id_al: np.zeros(5, dtype=np.float32) for id_al in todos_alumnos}
    
    # Llenar Notas (0-3)
    for _, row in df.iterrows():
        id_al = row["ID"]
        curso = row["CURSO"]
        if curso in idx_primer:
            # ✅ NORMALIZACIÓN DE NOTAS (/ 7.0)
            val = limpiar_nota(row["NOTA"], row["ESTADO_CURSO"]) / 7.0
            vectores[id_al][idx_primer[curso]] = val

    # Llenar Puntaje (4)
    for id_al in vectores:
        score = mapa_puntajes.get(id_al, np.nan)
        if pd.isna(score):
            vectores[id_al][4] = -1.0 # Sin puntaje (valor fuera de rango normal)
        else:
            # ✅ NORMALIZACIÓN DE PUNTAJE (/ Max)
            vectores[id_al][4] = score / max_score

    df_vectores = pd.DataFrame.from_dict(vectores, orient='index')
    return df_vectores

def obtener_ids_reprobados(ruta_target, cursos_eval):
    df = pd.read_csv(ruta_target, sep=';')
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df = df[df['CURSO'].isin(cursos_eval)]
    reprobados = set()
    for _, row in df.iterrows():
        estado = str(row['ESTADO_CURSO'])
        nota = row['NOTA']
        es_repro = False
        if "Reprobado" in estado: es_repro = True
        elif "Aprobado" not in estado:
            try: 
                if float(str(nota).replace(",", ".")) < 4.0: es_repro = True
            except: pass
        if es_repro: reprobados.add(row['ID'])
    return reprobados

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

# ENTRENAMIENTO BALANCEADO
def entrenar_predictor_balanceado(X_data, Y_data, ids_comunes, ids_reprobados_set, save_path):
    ids_train_val, ids_test = train_test_split(ids_comunes, test_size=0.20, random_state=42)
    ids_train, ids_val      = train_test_split(ids_train_val, test_size=0.15, random_state=42)
    
    id_to_idx = {uid: i for i, uid in enumerate(ids_comunes)}
    idxs_train = [id_to_idx[uid] for uid in ids_train]
    idxs_val   = [id_to_idx[uid] for uid in ids_val]
    idxs_test  = [id_to_idx[uid] for uid in ids_test]
    
    # Balanceo
    idxs_reprobados_train = [i for i, uid in zip(idxs_train, ids_train) if uid in ids_reprobados_set]
    n_apr = len(idxs_train) - len(idxs_reprobados_train)
    n_rep = len(idxs_reprobados_train)
    
    final_idxs_train = list(idxs_train)
    if n_rep > 0 and n_apr > n_rep:
        factor = n_apr // n_rep
        final_idxs_train += idxs_reprobados_train * factor
    
    X_train_t = torch.FloatTensor(X_data[final_idxs_train])
    Y_train_t = torch.FloatTensor(Y_data[final_idxs_train])
    X_val_t = torch.FloatTensor(X_data[idxs_val])
    Y_val_t = torch.FloatTensor(Y_data[idxs_val])
    X_test_t = torch.FloatTensor(X_data[idxs_test])
    Y_test_t = torch.FloatTensor(Y_data[idxs_test])
    
    predictor = EmbeddingPredictor(X_train_t.shape[1], Y_train_t.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(predictor.parameters(), lr=1e-3)
    best_loss, patience, counter = float('inf'), 200, 0

    for epoch in range(1000):
        predictor.train()
        optimizer.zero_grad()
        loss = criterion(predictor(X_train_t), Y_train_t)
        loss.backward(); optimizer.step()

        predictor.eval()
        with torch.no_grad():
            vloss = criterion(predictor(X_val_t), Y_val_t)

        if vloss.item() < best_loss - 1e-9:
            best_loss = vloss.item()
            torch.save(predictor.state_dict(), save_path)
            counter = 0
        else:
            counter += 1
            if counter >= patience: break

    predictor.load_state_dict(torch.load(save_path, map_location=DEVICE))
    predictor.eval()
    with torch.no_grad():
        test_mse = criterion(predictor(X_test_t), Y_test_t).item()
    return test_mse

# ================================================================
# 🔹 MAIN
# ================================================================
def main():
    os.makedirs(SAVE_BASE, exist_ok=True)
    
    print("--- Fase 0: Identificando Reprobados (Target) ---")
    ids_reprobados = obtener_ids_reprobados(RUTA_DF_TARGET, CURSOS_EVAL)
    print(f"   -> Alumnos que reprueban algo: {len(ids_reprobados)}")

    print("\n--- Fase 1: Cargando Vocabulario TuckER ---")
    alumnos_tucker = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    
    print("\n--- Fase 2: Preparando Vectores 5D (4 Notas + Puntaje) ---")
    # ✅ Llama a la función que normaliza notas y puntajes
    df_vectores = preparar_X_historico_5dim(RUTA_DF_INPUT, RUTA_PUNTAJES, CURSOS_PRIMER)
    
    vocab_completo = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(vocab_completo)}
    
    alumnos_comunes = sorted(set(df_vectores.index).intersection(vocab_completo))
    print(f"   -> Alumnos Comunes (Trainable): {len(alumnos_comunes)}")

    X_data_total = df_vectores.loc[alumnos_comunes].values.astype(np.float32)

    resumen = []
    print("\n=== Bucle por RDIM (Entrenamiento 5D Balanceado) ===")
    for rdim in RDIMS:
        tucker_folder = RUN_PREFIX.format(rdim=rdim)
        tucker_pt = os.path.join(RESULTS_BASE, tucker_folder, "best_model.pt")
        save_path = os.path.join(SAVE_BASE, f"best_predictor_dim5_rdim{rdim}_balanceado_5d.pt")

        print(f"\n>> rdim={rdim} | checkpoint: {tucker_pt}")
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ No existe checkpoint TuckER. Se omite.")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            idxs_tucker = [entity_idxs[a] for a in alumnos_comunes]
            Y_data_total = E[idxs_tucker].numpy()

            test_mse = entrenar_predictor_balanceado(
                X_data_total, Y_data_total, 
                alumnos_comunes, ids_reprobados, 
                save_path
            )
            
            print(f"   ✅ Guardado: {save_path}")
            print(f"   📏 MSE (test): {test_mse:.6f}")
            resumen.append((rdim, E.shape[1], len(alumnos_comunes), test_mse, save_path))
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            import traceback
            traceback.print_exc()
            continue

    if resumen:
        df_sum = pd.DataFrame(resumen, columns=["rdim","edim","n_alumnos","test_mse","ruta_modelo"])
        out_csv = os.path.join(SAVE_BASE, "resumen_nn_sem1_balanceado_5d.csv")
        df_sum.sort_values("rdim").to_csv(out_csv, index=False)
        print(f"\n=== Resumen guardado: {out_csv} ===")
        print(df_sum.sort_values("rdim").to_string(index=False))

if __name__ == "__main__":
    main()

--- Fase 0: Identificando Reprobados (Target) ---
   -> Alumnos que reprueban algo: 55

--- Fase 1: Cargando Vocabulario TuckER ---

--- Fase 2: Preparando Vectores 5D (4 Notas + Puntaje) ---
🔄 Construyendo vectores históricos de dimensión 5 (4 Notas + Puntaje)...
   Max Puntaje detectado para normalizar: 935.2
   -> Alumnos Comunes (Trainable): 790

=== Bucle por RDIM (Entrenamiento 5D Balanceado) ===

>> rdim=1 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim1_1000epochs_earlystopping_2019_2020patience400_balanceado_5d\best_model.pt
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d\best_predictor_dim5_rdim1_balanceado_5d.pt
   📏 MSE (test): 0.017833

>> rdim=2 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim2_1000epochs_earlystopping_2019_2020patience400_balanceado_5d\best_model.pt
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predict

# redes neruonales no balanceadas

In [11]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# CONFIGURACIÓN GENERAL
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# --- RUTAS ---
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado"
RESULTS_BASE    = r"C:\Users\56946\TuckER\results"

# Prefijo del TuckER 5D (Balanceado o no, según el .bat)
RUN_PREFIX      = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"

# ⚠️ GUARDAR EN CARPETA DE PREDICORES ESTÁNDAR 5D
SAVE_BASE       = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_standard"

# --- DATOS ---
RUTA_DF_INPUT  = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\df_20191.csv"
RUTA_DF_TARGET = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre\df_20192.csv"
RUTA_PUNTAJES  = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER  = ['MA1101', 'MA1001', 'FI1000', 'BT1211']
CURSOS_SEGUNDO = ['MA1002', 'MA1102', 'FI1100', 'CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = range(1, 17)

# =========================
# FUNCIONES AUXILIARES
# =========================
def get_vocab_from_data_dir(data_dir):
    entities = set()
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    h, r, t = line.strip().split()
                    entities.add(h.strip().upper())
    return sorted(list(entities))

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def cargar_E_weights(path_pt):
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    return sd["E.weight"].detach().cpu()

def limpiar_nota(nota_str, estado):
    if isinstance(estado, str) and "Reprobado" in estado: return 1.0
    try:
        if pd.isna(nota_str) or str(nota_str).strip() == "": return 0.0
        return float(str(nota_str).replace(",", "."))
    except: return 0.0

# VECTOR 5D (Notas + Puntaje)
def preparar_X_historico_5dim(ruta_input, ruta_puntajes, cursos_primer):
    print(f"🔄 Construyendo vectores históricos de dimensión 5 (4 Notas + Puntaje)...")
    
    df = pd.read_csv(ruta_input, sep=';')
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    todos_alumnos = df["ID"].unique()
    
    df_puntajes = pd.read_csv(ruta_puntajes, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    max_score = df_puntajes[df_puntajes["ID"].isin(todos_alumnos)]["PUNTAJE_PONDERADO"].max()
    if pd.isna(max_score): max_score = 850.0
    print(f"   Max Puntaje detectado: {max_score}")

    mapa_puntajes = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    idx_primer = {c: i for i, c in enumerate(cursos_primer)}
    vectores = {id_al: np.zeros(5, dtype=np.float32) for id_al in todos_alumnos}
    
    for _, row in df.iterrows():
        id_al = row["ID"]
        curso = row["CURSO"]
        if curso in idx_primer:
            val = limpiar_nota(row["NOTA"], row["ESTADO_CURSO"]) / 7.0
            vectores[id_al][idx_primer[curso]] = val

    for id_al in vectores:
        score = mapa_puntajes.get(id_al, np.nan)
        if pd.isna(score):
            vectores[id_al][4] = -1.0 
        else:
            vectores[id_al][4] = score / max_score

    return pd.DataFrame.from_dict(vectores, orient='index')

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

# ⚠️ ENTRENAMIENTO ESTÁNDAR (SIN BALANCEO)
def entrenar_predictor_estandar(X_data, Y_data, save_path):
    # Split aleatorio simple
    X_train_val, X_test, Y_train_val, Y_test = train_test_split(X_data, Y_data, test_size=0.20, random_state=42)
    X_train, X_val, Y_train, Y_val = train_test_split(X_train_val, Y_train_val, test_size=0.15, random_state=42)
    
    X_train_t = torch.FloatTensor(X_train)
    Y_train_t = torch.FloatTensor(Y_train)
    X_val_t   = torch.FloatTensor(X_val)
    Y_val_t   = torch.FloatTensor(Y_val)
    X_test_t  = torch.FloatTensor(X_test)
    Y_test_t  = torch.FloatTensor(Y_test)
    
    predictor = EmbeddingPredictor(X_train_t.shape[1], Y_train_t.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(predictor.parameters(), lr=1e-3)
    best_loss, patience, counter = float('inf'), 200, 0

    for epoch in range(1000):
        predictor.train()
        optimizer.zero_grad()
        loss = criterion(predictor(X_train_t), Y_train_t)
        loss.backward(); optimizer.step()

        predictor.eval()
        with torch.no_grad():
            vloss = criterion(predictor(X_val_t), Y_val_t)

        if vloss.item() < best_loss - 1e-9:
            best_loss = vloss.item()
            torch.save(predictor.state_dict(), save_path)
            counter = 0
        else:
            counter += 1
            if counter >= patience: break

    predictor.load_state_dict(torch.load(save_path, map_location=DEVICE))
    predictor.eval()
    with torch.no_grad():
        test_mse = criterion(predictor(X_test_t), Y_test_t).item()
    return test_mse

# ================================================================
# 🔹 MAIN
# ================================================================
def main():
    os.makedirs(SAVE_BASE, exist_ok=True)
    print(f"📂 Carpeta Salida: {SAVE_BASE}")

    print("\n--- Fase 1: Cargando Vocabulario TuckER ---")
    alumnos_tucker = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    
    print("\n--- Fase 2: Preparando Vectores 5D (4 Notas + Puntaje) ---")
    df_vectores = preparar_X_historico_5dim(RUTA_DF_INPUT, RUTA_PUNTAJES, CURSOS_PRIMER)
    
    vocab_completo = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(vocab_completo)}
    
    alumnos_comunes = sorted(set(df_vectores.index).intersection(vocab_completo))
    print(f"   -> Alumnos Comunes (Trainable): {len(alumnos_comunes)}")

    X_data_total = df_vectores.loc[alumnos_comunes].values.astype(np.float32)

    resumen = []
    print("\n=== Bucle por RDIM (Entrenamiento 5D ESTÁNDAR) ===")
    for rdim in RDIMS:
        tucker_folder = RUN_PREFIX.format(rdim=rdim)
        tucker_pt = os.path.join(RESULTS_BASE, tucker_folder, "best_model.pt")
        save_path = os.path.join(SAVE_BASE, f"best_predictor_dim5_rdim{rdim}_standard_5d.pt")

        print(f"\n>> rdim={rdim} | checkpoint: {tucker_pt}")
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ No existe checkpoint TuckER. Se omite.")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            idxs_tucker = [entity_idxs[a] for a in alumnos_comunes]
            Y_data_total = E[idxs_tucker].numpy()

            test_mse = entrenar_predictor_estandar(
                X_data_total, Y_data_total, 
                save_path
            )
            
            print(f"   ✅ Guardado: {save_path}")
            print(f"   📏 MSE (test): {test_mse:.6f}")
            resumen.append((rdim, E.shape[1], len(alumnos_comunes), test_mse, save_path))
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            import traceback
            traceback.print_exc()
            continue

    if resumen:
        df_sum = pd.DataFrame(resumen, columns=["rdim","edim","n_alumnos","test_mse","ruta_modelo"])
        out_csv = os.path.join(SAVE_BASE, "resumen_nn_sem1_standard_5d.csv")
        df_sum.sort_values("rdim").to_csv(out_csv, index=False)
        print(f"\n=== Resumen guardado: {out_csv} ===")
        print(df_sum.sort_values("rdim").to_string(index=False))

if __name__ == "__main__":
    main()

📂 Carpeta Salida: C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_standard

--- Fase 1: Cargando Vocabulario TuckER ---

--- Fase 2: Preparando Vectores 5D (4 Notas + Puntaje) ---
🔄 Construyendo vectores históricos de dimensión 5 (4 Notas + Puntaje)...
   Max Puntaje detectado: 935.2
   -> Alumnos Comunes (Trainable): 790

=== Bucle por RDIM (Entrenamiento 5D ESTÁNDAR) ===

>> rdim=1 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim1_1000epochs_earlystopping_2019_2020patience400_balanceado_5d\best_model.pt
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_standard\best_predictor_dim5_rdim1_standard_5d.pt
   📏 MSE (test): 0.018493

>> rdim=2 | checkpoint: C:\Users\56946\TuckER\results\Experimento_warm_start_rdim2_1000epochs_earlystopping_2019_2020patience400_balanceado_5d\best_model.pt
   ✅ Guardado: C:\Users\56946\TuckER\notebooks\Experimento_warm

# probar el modelo: Métricas guresas (balanceado + red balanceada)

In [4]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

# Dataset Multirrelacional
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado"

# Ruta de TuckER 5D Balanceado
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"
    r"\best_model.pt"
)

# Ruta de Predictores 5D
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_balanceados"
    r"\best_predictor_dim5_rdim{rdim}_balanceado_5d.pt"
)

# Datos de evaluación (2021)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")
# Archivo de puntajes (Necesario para la 5ta dimensión)
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones del modelo multirrelacional
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17) 

DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES Y MODELO
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe el predictor: {predictor_path}")
    state = torch.load(predictor_path, map_location=device)
    state = pick_state_dict(state)
    model.load_state_dict(state)
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe el modelo TuckER: {tucker_path}")
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h.strip().upper())
                        entities.add(t.strip().upper())
                        relations.add(r.strip())
    
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    
    return SimpleNamespace(
        entities=entities, 
        relations=relations_full,
        entity_idxs={e: i for i, e in enumerate(entities)}, 
        relation_idxs={r: i for i, r in enumerate(relations_full)}
    )

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ⚠️ VECTOR 5D (Notas + Puntaje)
def get_notes_vector_5d(csv_path, puntajes_map, max_score, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, len(cursos_primer)+1), dtype=torch.float32)
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    
    # 1. Notas (4 dims)
    # Rellenar con 0 si no hay nota
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    vec_notas = np.zeros(4, dtype=np.float32)
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            # Nota normalizada / 7.0
            vec_notas[idx_map[c]] = float(row["NOTA"]) / 7.0
            
    # 2. Puntaje (1 dim)
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val):
        puntaje_norm = -1.0
    else:
        puntaje_norm = score_val / max_score
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 AGREGACIÓN BINARIA
# ============================================================
def es_reprobado(relacion_str):
    return "reprueba" in relacion_str.lower()

# ============================================================
# 🔹 CARGA DATOS
# ============================================================
print("Cargando Dataframes 2021...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

# Cargar Puntajes
print("Cargando Puntajes...")
df_puntajes = pd.read_csv(RUTA_PUNTAJES, sep=';')
df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
# Mapa y Max Score (usamos el max global o el de la gen actual para consistencia)
max_score_global = df_puntajes["PUNTAJE_PONDERADO"].max()
if pd.isna(max_score_global): max_score_global = 850.0
puntajes_map = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()

for df in (df_20211, df_20212):
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(row):
    nota = pd.to_numeric(row['NOTA'], errors='coerce')
    estado = str(row['ESTADO_CURSO'])
    
    if "Reprobado" in estado: return "reprueba"
    if pd.isna(nota): return "reprueba" 
    
    if nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval.apply(categoria_relacion, axis=1)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- INICIANDO EVALUACIÓN MULTIRRELACIONAL (5D - NOTAS + PUNTAJE) ---")
    
    vocab = get_vocab_manual(DATA_DIR)
    
    for rdim in RDIMS:
        print(f"\n============================== Evaluando rdim={rdim} ==============================")
        
        tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
        predictor_path = PRED_DIR_FMT.format(rdim=rdim)

        if not os.path.exists(tucker_path):
            print(f"⏩ Saltando (Falta TuckER): {tucker_path}")
            continue
        if not os.path.exists(predictor_path):
            print(f"⏩ Saltando (Falta Predictor): {predictor_path}")
            continue

        try:
            E, R, W = load_tucker_weights(tucker_path, DEVICE)
            d1 = E.shape[1]

            rel2idx_map = {}
            for rel_hint in RELACIONES:
                idx = find_relation(vocab.relations, vocab.relation_idxs, hint=rel_hint)
                if idx is not None: rel2idx_map[rel_hint] = idx

            # ⚠️ INPUT SIZE = 5
            predictor = load_predictor(predictor_path, input_size=5, out_dim=d1, device="cpu")

            # Precalcular embeddings (Usando función 5D)
            ehat_cache = {}
            for aid in df_eval["ID"].unique():
                x = get_notes_vector_5d(CSV_20211, puntajes_map, max_score_global, aid, CURSOS_PRIMER)
                with torch.no_grad(): ehat_cache[aid] = predictor(x).squeeze(0)

            y_true_bin = []
            y_pred_bin = []
            
            # Matriz de Confusión Detallada (Multiclase)
            y_true_multi = []
            y_pred_multi = []

            for _, row in df_eval.iterrows():
                aid = row["ID"]
                curso = row["CURSO"]
                true_rel = row["RELACION_REAL"]

                if curso not in vocab.entity_idxs: continue
                
                t_idx = vocab.entity_idxs[curso]
                e_hat = ehat_cache[aid]
                
                # Scores
                scores = {}
                for r_name, r_idx in rel2idx_map.items():
                    scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
                
                # Ganadora
                pred_rel = max(scores, key=scores.get)
                
                # Multiclase
                y_true_multi.append(true_rel)
                y_pred_multi.append(pred_rel)
                
                # Binario
                real_bin = 1 if es_reprobado(true_rel) else 0
                pred_bin = 1 if es_reprobado(pred_rel) else 0
                
                y_true_bin.append(real_bin)
                y_pred_bin.append(pred_bin)

            # --- Métricas Binarias (Reprueba vs Resto) ---
            tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
            
            total_reales_reprobados = tp + fn
            total_predichos_reprobados = tp + fp
            
            recall = tp / total_reales_reprobados if total_reales_reprobados > 0 else 0
            precision = tp / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
            accuracy = (tp + tn) / len(y_true_bin)
            
            print(f"📊 Resultados Binarios (Focus: Reprobación)")
            print("-" * 60)
            print(f"Accuracy Global         : {accuracy:.4f}")
            print(f"✅ RECALL (Sensibilidad) : {recall:.4f} ({tp}/{total_reales_reprobados})")
            print(f"🎯 PRECISION             : {precision:.4f}")
            print(f"Matriz: [TP={tp} FN={fn}] | [FP={fp} TN={tn}]")
            
            print("\n📊 Reporte Detallado por Relación")
            print(classification_report(y_true_multi, y_pred_multi, labels=RELACIONES, digits=4))

        except Exception as e:
            print(f"❌ Error procesando rdim={rdim}: {e}")
            import traceback
            traceback.print_exc()

if __name__ == "__main__":
    main()

Cargando Dataframes 2021...
Cargando Puntajes...
Evaluaciones totales: 3023

--- INICIANDO EVALUACIÓN MULTIRRELACIONAL (5D - NOTAS + PUNTAJE) ---

============================== Evaluando rdim=1 ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.7142
✅ RECALL (Sensibilidad) : 0.0944 (22/233)
🎯 PRECISION             : 0.0326
Matriz: [TP=22 FN=211] | [FP=653 TN=2137]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0326    0.0944    0.0485       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.0000    0.0000    0.0000      1345
 aprueba_6_7     0.2781    0.7506    0.4058       870

    accuracy                         0.2233      3023
   macro avg     0.0777    0.2112    0.1136      3023
weighted avg     0.0826    0.2233    0.1205      3023


============================== Evaluando rdim=2 ==

C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.4939
✅ RECALL (Sensibilidad) : 0.6781 (158/233)
🎯 PRECISION             : 0.0980
Matriz: [TP=158 FN=75] | [FP=1455 TN=1335]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0980    0.6781    0.1712       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.5106    0.5353    0.5227      1345
 aprueba_6_7     0.0000    0.0000    0.0000       870

    accuracy                         0.2904      3023
   macro avg     0.1521    0.3034    0.1735      3023
weighted avg     0.2347    0.2904    0.2457      3023


============================== Evaluando rdim=3 ==============================


C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.9229
✅ RECALL (Sensibilidad) : 0.0000 (0/233)
🎯 PRECISION             : 0.0000
Matriz: [TP=0 FN=233] | [FP=0 TN=2790]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0000    0.0000    0.0000       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.0000    0.0000    0.0000      1345
 aprueba_6_7     0.2878    1.0000    0.4470       870

    accuracy                         0.2878      3023
   macro avg     0.0719    0.2500    0.1117      3023
weighted avg     0.0828    0.2878    0.1286      3023


============================== Evaluando rdim=4 ==============================


...

C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.7102
✅ RECALL (Sensibilidad) : 0.2833 (66/233)
🎯 PRECISION             : 0.0852
Matriz: [TP=66 FN=167] | [FP=709 TN=2081]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0852    0.2833    0.1310       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.4399    0.7346    0.5503      1345
 aprueba_6_7     0.0000    0.0000    0.0000       870

    accuracy                         0.3487      3023
   macro avg     0.1313    0.2545    0.1703      3023
weighted avg     0.2023    0.3487    0.2549      3023


============================== Evaluando rdim=16 ==============================


C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.0771
✅ RECALL (Sensibilidad) : 1.0000 (233/233)
🎯 PRECISION             : 0.0771
Matriz: [TP=233 FN=0] | [FP=2790 TN=0]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0771    1.0000    0.1431       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.0000    0.0000    0.0000      1345
 aprueba_6_7     0.0000    0.0000    0.0000       870

    accuracy                         0.0771      3023
   macro avg     0.0193    0.2500    0.0358      3023
weighted avg     0.0059    0.0771    0.0110      3023



C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [9]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

# ⚠️ CORRECCIÓN AQUÍ: Agregamos la barra final para que load_data.py funcione
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado/"

# Rutas de TuckER 5D Balanceado
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"
    r"\best_model.pt"
)

# Rutas de Predictores 5D
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_balanceados"
    r"\best_predictor_dim5_rdim{rdim}_balanceado_5d.pt"
)

# Datos
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")
# Archivo de Puntajes
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones a rankear
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17)

DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES
# ============================================================

sys.path.append(r"C:\Users\56946\TuckER")
from load_data import Data

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe: {predictor_path}")
    state = torch.load(predictor_path, map_location=device)
    model.load_state_dict(pick_state_dict(state))
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe: {tucker_path}")
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def build_vocab(data_dir, reverse=True):
    d = Data(data_dir=data_dir, reverse=reverse)
    entities, relations = d.entities, d.relations
    ent2idx = {e:i for i,e in enumerate(entities)}
    rel2idx = {r:i for i,r in enumerate(relations)}
    return SimpleNamespace(entities=entities, relations=relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ⚠️ VECTOR 5D (Notas + Puntaje)
def get_notes_vector_5d(csv_path, puntajes_map, max_score, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, 5), dtype=torch.float32) # Dim 5
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    vec_notas = np.zeros(4, dtype=np.float32)
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            vec_notas[idx_map[c]] = float(row["NOTA"]) / 7.0
            
    # Puntaje
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val): puntaje_norm = -1.0
    else: puntaje_norm = score_val / max_score
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 CARGA Y PREPARACIÓN DE DATOS
# ============================================================

print("Cargando Dataframes...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

# Cargar Puntajes
print("Cargando Puntajes...")
if os.path.exists(RUTA_PUNTAJES):
    df_puntajes = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    # Max Score Global
    max_score_global = df_puntajes["PUNTAJE_PONDERADO"].max()
    if pd.isna(max_score_global): max_score_global = 850.0
    puntajes_map = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
else:
    print("⚠️ No se encontró archivo de puntajes. Usando -1 por defecto.")
    puntajes_map = {}
    max_score_global = 850.0

for df in (df_20211, df_20212):
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()
print(f"🔹 {len(alumnos_validos)} alumnos válidos 20211.")

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(nota):
    if pd.isna(nota) or nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval["NOTA"].apply(categoria_relacion)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 LOOP EVALUACIÓN
# ============================================================

resumen_metricas = []

for rdim in RDIMS:
    print(f"\n============================== Evaluando rdim={rdim} (5D) ==============================")

    tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
    predictor_path = PRED_DIR_FMT.format(rdim=rdim)

    if not os.path.exists(tucker_path):
        print(f"⚠️ Faltan archivos TuckER")
        continue

    d = build_vocab(DATA_DIR, reverse=True)
    E, R, W = load_tucker_weights(tucker_path, DEVICE)
    d1 = E.shape[1]

    rel2idx_map = {}
    for rel_hint in RELACIONES:
        idx = find_relation(d.relations, d.relation_idxs, hint=rel_hint)
        if idx is not None: rel2idx_map[rel_hint] = idx

    try:
        # ⚠️ Input Size = 5
        predictor = load_predictor(predictor_path, input_size=5, out_dim=d1, device="cpu")
    except Exception as e:
        print(f"⚠️ Error cargando predictor: {e}")
        continue

    ehat_by_alumno = {}
    for aid in df_eval["ID"].unique():
        # Usar función 5D
        x = get_notes_vector_5d(CSV_20211, puntajes_map, max_score_global, aid, CURSOS_PRIMER)
        with torch.no_grad(): ehat_by_alumno[aid] = predictor(x).squeeze(0)

    # 1. Binarización para cálculo "Reprueba vs El Resto"
    y_true_bin = []
    y_pred_bin = []

    for _, row in df_eval.iterrows():
        aid, curso, true_rel = row["ID"], row["CURSO"], row["RELACION_REAL"]
        if curso not in d.entity_idxs: continue
        
        t_idx = d.entity_idxs[curso]
        e_hat = ehat_by_alumno[aid]

        scores = {}
        for r_name, r_idx in rel2idx_map.items():
            scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
        
        pred_rel = max(scores, key=scores.get)
        
        # Lógica Binaria (Reprueba = 1)
        real_bin = 1 if "reprueba" in true_rel.lower() else 0
        pred_bin = 1 if "reprueba" in pred_rel.lower() else 0
        
        y_true_bin.append(real_bin)
        y_pred_bin.append(pred_bin)

    if not y_true_bin: continue

    # Métricas Binarias
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
    
    total_reales_reprobados = tp + fn
    total_predichos_reprobados = tp + fp
    
    recall = tp / total_reales_reprobados if total_reales_reprobados > 0 else 0
    precision = tp / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
    accuracy = (tp + tn) / len(y_true_bin)
    
    print(f"📊 Resultados Binarios (Focus: Reprobación)")
    print("-" * 60)
    print(f"Accuracy Global         : {accuracy:.4f}")
    print(f"✅ RECALL (Sensibilidad) : {recall:.4f} ({tp}/{total_reales_reprobados})")
    print(f"🎯 PRECISION             : {precision:.4f}")
    print(f"Matriz: [TP={tp} FN={fn}] | [FP={fp} TN={tn}]")

Cargando Dataframes...
Cargando Puntajes...
🔹 830 alumnos válidos 20211.
Evaluaciones totales: 3023

============================== Evaluando rdim=1 (5D) ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.7142
✅ RECALL (Sensibilidad) : 0.0944 (22/233)
🎯 PRECISION             : 0.0326
Matriz: [TP=22 FN=211] | [FP=653 TN=2137]

============================== Evaluando rdim=2 (5D) ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.4939
✅ RECALL (Sensibilidad) : 0.6781 (158/233)
🎯 PRECISION             : 0.0980
Matriz: [TP=158 FN=75] | [FP=1455 TN=1335]

============================== Evaluando rdim=3 (5D) ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.922

# probar el modelo: Métricas guresas (balanceado + red sin balancear)

In [12]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

# ⚠️ CORRECCIÓN AQUÍ: Agregamos la barra final para que load_data.py funcione
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado/"

# Rutas de TuckER 5D Balanceado
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"
    r"\best_model.pt"
)

# Rutas de Predictores 5D
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_standard"
    r"\best_predictor_dim5_rdim{rdim}_standard_5d.pt"
)

# Datos
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")
# Archivo de Puntajes
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones a rankear
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17)

DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES
# ============================================================

sys.path.append(r"C:\Users\56946\TuckER")
from load_data import Data

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe: {predictor_path}")
    state = torch.load(predictor_path, map_location=device)
    model.load_state_dict(pick_state_dict(state))
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe: {tucker_path}")
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def build_vocab(data_dir, reverse=True):
    d = Data(data_dir=data_dir, reverse=reverse)
    entities, relations = d.entities, d.relations
    ent2idx = {e:i for i,e in enumerate(entities)}
    rel2idx = {r:i for i,r in enumerate(relations)}
    return SimpleNamespace(entities=entities, relations=relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ⚠️ VECTOR 5D (Notas + Puntaje)
def get_notes_vector_5d(csv_path, puntajes_map, max_score, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, 5), dtype=torch.float32) # Dim 5
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    vec_notas = np.zeros(4, dtype=np.float32)
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            vec_notas[idx_map[c]] = float(row["NOTA"]) / 7.0
            
    # Puntaje
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val): puntaje_norm = -1.0
    else: puntaje_norm = score_val / max_score
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 CARGA Y PREPARACIÓN DE DATOS
# ============================================================

print("Cargando Dataframes...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

# Cargar Puntajes
print("Cargando Puntajes...")
if os.path.exists(RUTA_PUNTAJES):
    df_puntajes = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    # Max Score Global
    max_score_global = df_puntajes["PUNTAJE_PONDERADO"].max()
    if pd.isna(max_score_global): max_score_global = 850.0
    puntajes_map = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
else:
    print("⚠️ No se encontró archivo de puntajes. Usando -1 por defecto.")
    puntajes_map = {}
    max_score_global = 850.0

for df in (df_20211, df_20212):
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()
print(f"🔹 {len(alumnos_validos)} alumnos válidos 20211.")

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(nota):
    if pd.isna(nota) or nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval["NOTA"].apply(categoria_relacion)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 LOOP EVALUACIÓN
# ============================================================

resumen_metricas = []

for rdim in RDIMS:
    print(f"\n============================== Evaluando rdim={rdim} (5D) ==============================")

    tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
    predictor_path = PRED_DIR_FMT.format(rdim=rdim)

    if not os.path.exists(tucker_path):
        print(f"⚠️ Faltan archivos TuckER")
        continue

    d = build_vocab(DATA_DIR, reverse=True)
    E, R, W = load_tucker_weights(tucker_path, DEVICE)
    d1 = E.shape[1]

    rel2idx_map = {}
    for rel_hint in RELACIONES:
        idx = find_relation(d.relations, d.relation_idxs, hint=rel_hint)
        if idx is not None: rel2idx_map[rel_hint] = idx

    try:
        # ⚠️ Input Size = 5
        predictor = load_predictor(predictor_path, input_size=5, out_dim=d1, device="cpu")
    except Exception as e:
        print(f"⚠️ Error cargando predictor: {e}")
        continue

    ehat_by_alumno = {}
    for aid in df_eval["ID"].unique():
        # Usar función 5D
        x = get_notes_vector_5d(CSV_20211, puntajes_map, max_score_global, aid, CURSOS_PRIMER)
        with torch.no_grad(): ehat_by_alumno[aid] = predictor(x).squeeze(0)

    # 1. Binarización para cálculo "Reprueba vs El Resto"
    y_true_bin = []
    y_pred_bin = []

    for _, row in df_eval.iterrows():
        aid, curso, true_rel = row["ID"], row["CURSO"], row["RELACION_REAL"]
        if curso not in d.entity_idxs: continue
        
        t_idx = d.entity_idxs[curso]
        e_hat = ehat_by_alumno[aid]

        scores = {}
        for r_name, r_idx in rel2idx_map.items():
            scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
        
        pred_rel = max(scores, key=scores.get)
        
        # Lógica Binaria (Reprueba = 1)
        real_bin = 1 if "reprueba" in true_rel.lower() else 0
        pred_bin = 1 if "reprueba" in pred_rel.lower() else 0
        
        y_true_bin.append(real_bin)
        y_pred_bin.append(pred_bin)

    if not y_true_bin: continue

    # Métricas Binarias
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
    
    total_reales_reprobados = tp + fn
    total_predichos_reprobados = tp + fp
    
    recall = tp / total_reales_reprobados if total_reales_reprobados > 0 else 0
    precision = tp / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
    accuracy = (tp + tn) / len(y_true_bin)
    
    print(f"📊 Resultados Binarios (Focus: Reprobación)")
    print("-" * 60)
    print(f"Accuracy Global         : {accuracy:.4f}")
    print(f"✅ RECALL (Sensibilidad) : {recall:.4f} ({tp}/{total_reales_reprobados})")
    print(f"🎯 PRECISION             : {precision:.4f}")
    print(f"Matriz: [TP={tp} FN={fn}] | [FP={fp} TN={tn}]")

Cargando Dataframes...
Cargando Puntajes...
🔹 830 alumnos válidos 20211.
Evaluaciones totales: 3023

============================== Evaluando rdim=1 (5D) ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.7142
✅ RECALL (Sensibilidad) : 0.0944 (22/233)
🎯 PRECISION             : 0.0326
Matriz: [TP=22 FN=211] | [FP=653 TN=2137]

============================== Evaluando rdim=2 (5D) ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.4939
✅ RECALL (Sensibilidad) : 0.6781 (158/233)
🎯 PRECISION             : 0.0980
Matriz: [TP=158 FN=75] | [FP=1455 TN=1335]

============================== Evaluando rdim=3 (5D) ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.922

In [13]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

# Dataset Multirrelacional
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado"

# Ruta de TuckER 5D Balanceado
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"
    r"\best_model.pt"
)

# Ruta de Predictores 5D
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_standard"
    r"\best_predictor_dim5_rdim{rdim}_standard_5d.pt"
)

# Datos de evaluación (2021)
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")
# Archivo de puntajes (Necesario para la 5ta dimensión)
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones del modelo multirrelacional
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17) 

DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES Y MODELO
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe el predictor: {predictor_path}")
    state = torch.load(predictor_path, map_location=device)
    state = pick_state_dict(state)
    model.load_state_dict(state)
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe el modelo TuckER: {tucker_path}")
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h.strip().upper())
                        entities.add(t.strip().upper())
                        relations.add(r.strip())
    
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    
    return SimpleNamespace(
        entities=entities, 
        relations=relations_full,
        entity_idxs={e: i for i, e in enumerate(entities)}, 
        relation_idxs={r: i for i, r in enumerate(relations_full)}
    )

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ⚠️ VECTOR 5D (Notas + Puntaje)
def get_notes_vector_5d(csv_path, puntajes_map, max_score, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, len(cursos_primer)+1), dtype=torch.float32)
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    
    # 1. Notas (4 dims)
    # Rellenar con 0 si no hay nota
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    vec_notas = np.zeros(4, dtype=np.float32)
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            # Nota normalizada / 7.0
            vec_notas[idx_map[c]] = float(row["NOTA"]) / 7.0
            
    # 2. Puntaje (1 dim)
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val):
        puntaje_norm = -1.0
    else:
        puntaje_norm = score_val / max_score
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 AGREGACIÓN BINARIA
# ============================================================
def es_reprobado(relacion_str):
    return "reprueba" in relacion_str.lower()

# ============================================================
# 🔹 CARGA DATOS
# ============================================================
print("Cargando Dataframes 2021...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

# Cargar Puntajes
print("Cargando Puntajes...")
df_puntajes = pd.read_csv(RUTA_PUNTAJES, sep=';')
df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
# Mapa y Max Score (usamos el max global o el de la gen actual para consistencia)
max_score_global = df_puntajes["PUNTAJE_PONDERADO"].max()
if pd.isna(max_score_global): max_score_global = 850.0
puntajes_map = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()

for df in (df_20211, df_20212):
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(row):
    nota = pd.to_numeric(row['NOTA'], errors='coerce')
    estado = str(row['ESTADO_CURSO'])
    
    if "Reprobado" in estado: return "reprueba"
    if pd.isna(nota): return "reprueba" 
    
    if nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval.apply(categoria_relacion, axis=1)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- INICIANDO EVALUACIÓN MULTIRRELACIONAL (5D - NOTAS + PUNTAJE) ---")
    
    vocab = get_vocab_manual(DATA_DIR)
    
    for rdim in RDIMS:
        print(f"\n============================== Evaluando rdim={rdim} ==============================")
        
        tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
        predictor_path = PRED_DIR_FMT.format(rdim=rdim)

        if not os.path.exists(tucker_path):
            print(f"⏩ Saltando (Falta TuckER): {tucker_path}")
            continue
        if not os.path.exists(predictor_path):
            print(f"⏩ Saltando (Falta Predictor): {predictor_path}")
            continue

        try:
            E, R, W = load_tucker_weights(tucker_path, DEVICE)
            d1 = E.shape[1]

            rel2idx_map = {}
            for rel_hint in RELACIONES:
                idx = find_relation(vocab.relations, vocab.relation_idxs, hint=rel_hint)
                if idx is not None: rel2idx_map[rel_hint] = idx

            # ⚠️ INPUT SIZE = 5
            predictor = load_predictor(predictor_path, input_size=5, out_dim=d1, device="cpu")

            # Precalcular embeddings (Usando función 5D)
            ehat_cache = {}
            for aid in df_eval["ID"].unique():
                x = get_notes_vector_5d(CSV_20211, puntajes_map, max_score_global, aid, CURSOS_PRIMER)
                with torch.no_grad(): ehat_cache[aid] = predictor(x).squeeze(0)

            y_true_bin = []
            y_pred_bin = []
            
            # Matriz de Confusión Detallada (Multiclase)
            y_true_multi = []
            y_pred_multi = []

            for _, row in df_eval.iterrows():
                aid = row["ID"]
                curso = row["CURSO"]
                true_rel = row["RELACION_REAL"]

                if curso not in vocab.entity_idxs: continue
                
                t_idx = vocab.entity_idxs[curso]
                e_hat = ehat_cache[aid]
                
                # Scores
                scores = {}
                for r_name, r_idx in rel2idx_map.items():
                    scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
                
                # Ganadora
                pred_rel = max(scores, key=scores.get)
                
                # Multiclase
                y_true_multi.append(true_rel)
                y_pred_multi.append(pred_rel)
                
                # Binario
                real_bin = 1 if es_reprobado(true_rel) else 0
                pred_bin = 1 if es_reprobado(pred_rel) else 0
                
                y_true_bin.append(real_bin)
                y_pred_bin.append(pred_bin)

            # --- Métricas Binarias (Reprueba vs Resto) ---
            tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
            
            total_reales_reprobados = tp + fn
            total_predichos_reprobados = tp + fp
            
            recall = tp / total_reales_reprobados if total_reales_reprobados > 0 else 0
            precision = tp / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
            accuracy = (tp + tn) / len(y_true_bin)
            
            print(f"📊 Resultados Binarios (Focus: Reprobación)")
            print("-" * 60)
            print(f"Accuracy Global         : {accuracy:.4f}")
            print(f"✅ RECALL (Sensibilidad) : {recall:.4f} ({tp}/{total_reales_reprobados})")
            print(f"🎯 PRECISION             : {precision:.4f}")
            print(f"Matriz: [TP={tp} FN={fn}] | [FP={fp} TN={tn}]")
            
            print("\n📊 Reporte Detallado por Relación")
            print(classification_report(y_true_multi, y_pred_multi, labels=RELACIONES, digits=4))

        except Exception as e:
            print(f"❌ Error procesando rdim={rdim}: {e}")
            import traceback
            traceback.print_exc()

if __name__ == "__main__":
    main()

Cargando Dataframes 2021...
Cargando Puntajes...
Evaluaciones totales: 3023

--- INICIANDO EVALUACIÓN MULTIRRELACIONAL (5D - NOTAS + PUNTAJE) ---

============================== Evaluando rdim=1 ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.7142
✅ RECALL (Sensibilidad) : 0.0944 (22/233)
🎯 PRECISION             : 0.0326
Matriz: [TP=22 FN=211] | [FP=653 TN=2137]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0326    0.0944    0.0485       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.0000    0.0000    0.0000      1345
 aprueba_6_7     0.2781    0.7506    0.4058       870

    accuracy                         0.2233      3023
   macro avg     0.0777    0.2112    0.1136      3023
weighted avg     0.0826    0.2233    0.1205      3023


============================== Evaluando rdim=2 ==

C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.4939
✅ RECALL (Sensibilidad) : 0.6781 (158/233)
🎯 PRECISION             : 0.0980
Matriz: [TP=158 FN=75] | [FP=1455 TN=1335]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0980    0.6781    0.1712       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.5106    0.5353    0.5227      1345
 aprueba_6_7     0.0000    0.0000    0.0000       870

    accuracy                         0.2904      3023
   macro avg     0.1521    0.3034    0.1735      3023
weighted avg     0.2347    0.2904    0.2457      3023


============================== Evaluando rdim=3 ==============================


C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.9229
✅ RECALL (Sensibilidad) : 0.0000 (0/233)
🎯 PRECISION             : 0.0000
Matriz: [TP=0 FN=233] | [FP=0 TN=2790]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0000    0.0000    0.0000       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.0000    0.0000    0.0000      1345
 aprueba_6_7     0.2878    1.0000    0.4470       870

    accuracy                         0.2878      3023
   macro avg     0.0719    0.2500    0.1117      3023
weighted avg     0.0828    0.2878    0.1286      3023


============================== Evaluando rdim=4 ==============================


...

C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.7106
✅ RECALL (Sensibilidad) : 0.2833 (66/233)
🎯 PRECISION             : 0.0853
Matriz: [TP=66 FN=167] | [FP=708 TN=2082]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0853    0.2833    0.1311       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.4398    0.7353    0.5504      1345
 aprueba_6_7     0.0000    0.0000    0.0000       870

    accuracy                         0.3490      3023
   macro avg     0.1313    0.2546    0.1704      3023
weighted avg     0.2022    0.3490    0.2550      3023


============================== Evaluando rdim=16 ==============================


C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.0771
✅ RECALL (Sensibilidad) : 1.0000 (233/233)
🎯 PRECISION             : 0.0771
Matriz: [TP=233 FN=0] | [FP=2790 TN=0]

📊 Reporte Detallado por Relación
              precision    recall  f1-score   support

    reprueba     0.0771    1.0000    0.1431       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.0000    0.0000    0.0000      1345
 aprueba_6_7     0.0000    0.0000    0.0000       870

    accuracy                         0.0771      3023
   macro avg     0.0193    0.2500    0.0358      3023
weighted avg     0.0059    0.0771    0.0110      3023



C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\56946\anaconda3\envs\tucker_env\lib\site-packages\sklearn\metrics\_classification.py:1248: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [16]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN DE RUTAS
# ============================================================

# Dataset Multirrelacional
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado"

# Ruta de TuckER 5D Balanceado
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"
    r"\best_model.pt"
)

# ⚠️ RUTA CORREGIDA: Predictores 5D Balanceados
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_balanceados"
    r"\best_predictor_dim5_rdim{rdim}_balanceado_5d.pt"
)

# Datos Base
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211 = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212 = os.path.join(BASE_PATH, "df_20212.csv")

# Archivo de Puntajes (Necesario para la 5ta dimensión)
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Cursos
CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones del modelo
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17) 
DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe: {predictor_path}")
    state = torch.load(predictor_path, map_location=device)
    model.load_state_dict(pick_state_dict(state))
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe: {tucker_path}")
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def get_vocab_manual(data_dir):
    entities, relations = set(), set()
    nombres = ['train.txt', 'train_balanceado.txt', 'valid.txt', 'test.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h.strip().upper())
                        entities.add(t.strip().upper())
                        relations.add(r.strip())
    
    entities = sorted(list(entities))
    relations = sorted(list(relations))
    relations_full = sorted(list(set(relations + [r + "_reverse" for r in relations])))
    
    return SimpleNamespace(
        entities=entities, 
        relations=relations_full,
        entity_idxs={e: i for i, e in enumerate(entities)}, 
        relation_idxs={r: i for i, r in enumerate(relations_full)}
    )

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ⚠️ VECTOR 5D (Notas + Puntaje)
def get_notes_vector_5d(csv_path, puntajes_map, max_score, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, 5), dtype=torch.float32)
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    vec_notas = np.zeros(4, dtype=np.float32)
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            vec_notas[idx_map[c]] = float(row["NOTA"]) / 7.0
            
    # Puntaje
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val): puntaje_norm = -1.0
    else: puntaje_norm = score_val / max_score
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 CARGA DATOS
# ============================================================
print("Cargando Dataframes 2021...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

# Cargar Puntajes
print("Cargando Puntajes...")
if os.path.exists(RUTA_PUNTAJES):
    df_puntajes = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    max_score_global = df_puntajes["PUNTAJE_PONDERADO"].max()
    if pd.isna(max_score_global): max_score_global = 850.0
    puntajes_map = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
else:
    print("⚠️ No se encontró archivo de puntajes. Usando default.")
    puntajes_map = {}
    max_score_global = 850.0

for df in (df_20211, df_20212):
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(row):
    nota = pd.to_numeric(row['NOTA'], errors='coerce')
    estado = str(row['ESTADO_CURSO'])
    
    if "Reprobado" in estado: return "reprueba"
    if pd.isna(nota): return "reprueba" 
    
    if nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval.apply(categoria_relacion, axis=1)

def es_reprobado(relacion_str):
    return "reprueba" in relacion_str.lower()

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print(f"\n--- EVALUACIÓN 5D BALANCEADA (Total Eval: {len(df_eval)}) ---")
    
    vocab = get_vocab_manual(DATA_DIR)
    
    for rdim in RDIMS:
        print(f"\n============================== Evaluando rdim={rdim} ==============================")
        
        tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
        predictor_path = PRED_DIR_FMT.format(rdim=rdim)

        if not os.path.exists(tucker_path):
            print(f"⏩ Saltando (Falta TuckER): {tucker_path}")
            continue
        if not os.path.exists(predictor_path):
            print(f"⏩ Saltando (Falta Predictor): {predictor_path}")
            continue

        try:
            E, R, W = load_tucker_weights(tucker_path, DEVICE)
            d1 = E.shape[1]

            rel2idx_map = {}
            for rel_hint in RELACIONES:
                idx = find_relation(vocab.relations, vocab.relation_idxs, hint=rel_hint)
                if idx is not None: rel2idx_map[rel_hint] = idx

            # ⚠️ INPUT SIZE = 5 (Notas + Puntaje)
            predictor = load_predictor(predictor_path, input_size=5, out_dim=d1, device="cpu")

            ehat_cache = {}
            for aid in df_eval["ID"].unique():
                x = get_notes_vector_5d(CSV_20211, puntajes_map, max_score_global, aid, CURSOS_PRIMER)
                with torch.no_grad(): ehat_cache[aid] = predictor(x).squeeze(0)

            y_true_bin = []
            y_pred_bin = []
            y_true_multi = []
            y_pred_multi = []

            for _, row in df_eval.iterrows():
                aid = row["ID"]
                curso = row["CURSO"]
                true_rel = row["RELACION_REAL"]

                if curso not in vocab.entity_idxs: continue
                
                t_idx = vocab.entity_idxs[curso]
                e_hat = ehat_cache[aid]
                
                scores = {}
                for r_name, r_idx in rel2idx_map.items():
                    scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
                
                pred_rel = max(scores, key=scores.get)
                
                # Guardar para binario y multi
                y_true_multi.append(true_rel)
                y_pred_multi.append(pred_rel)
                
                real_bin = 1 if es_reprobado(true_rel) else 0
                pred_bin = 1 if es_reprobado(pred_rel) else 0
                
                y_true_bin.append(real_bin)
                y_pred_bin.append(pred_bin)

            # Métricas
            tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
            total_reales_reprobados = tp + fn
            total_predichos_reprobados = tp + fp
            
            recall = tp / total_reales_reprobados if total_reales_reprobados > 0 else 0
            precision = tp / total_predichos_reprobados if total_predichos_reprobados > 0 else 0
            accuracy = (tp + tn) / len(y_true_bin)
            
            print(f"📊 Resultados Binarios (Focus: Reprobación)")
            print("-" * 60)
            print(f"Accuracy Global         : {accuracy:.4f}")
            print(f"✅ RECALL (Sensibilidad) : {recall:.4f} ({tp}/{total_reales_reprobados})")
            print(f"🎯 PRECISION             : {precision:.4f}")
            print(f"Matriz: [TP={tp} FN={fn}] | [FP={fp} TN={tn}]")
            
            print("\n📊 Detalle Multiclase:")
            print(classification_report(y_true_multi, y_pred_multi, labels=RELACIONES, digits=4, zero_division=0))

        except Exception as e:
            print(f"❌ Error rdim={rdim}: {e}")
            import traceback
            traceback.print_exc()

if __name__ == "__main__":
    main()

Cargando Dataframes 2021...
Cargando Puntajes...

--- EVALUACIÓN 5D BALANCEADA (Total Eval: 3023) ---

============================== Evaluando rdim=1 ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.7142
✅ RECALL (Sensibilidad) : 0.0944 (22/233)
🎯 PRECISION             : 0.0326
Matriz: [TP=22 FN=211] | [FP=653 TN=2137]

📊 Detalle Multiclase:
              precision    recall  f1-score   support

    reprueba     0.0326    0.0944    0.0485       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.0000    0.0000    0.0000      1345
 aprueba_6_7     0.2781    0.7506    0.4058       870

    accuracy                         0.2233      3023
   macro avg     0.0777    0.2112    0.1136      3023
weighted avg     0.0826    0.2233    0.1205      3023


============================== Evaluando rdim=2 ==============================
📊 Resultados Binarios (Foc

📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.6953
✅ RECALL (Sensibilidad) : 0.1974 (46/233)
🎯 PRECISION             : 0.0590
Matriz: [TP=46 FN=187] | [FP=734 TN=2056]

📊 Detalle Multiclase:
              precision    recall  f1-score   support

    reprueba     0.0590    0.1974    0.0908       233
 aprueba_4_5     0.0000    0.0000    0.0000       575
 aprueba_5_6     0.4307    0.7182    0.5385      1345
 aprueba_6_7     0.0000    0.0000    0.0000       870

    accuracy                         0.3348      3023
   macro avg     0.1224    0.2289    0.1573      3023
weighted avg     0.1962    0.3348    0.2466      3023


============================== Evaluando rdim=13 ==============================
📊 Resultados Binarios (Focus: Reprobación)
------------------------------------------------------------
Accuracy Global         : 0.2858
✅ RECALL (Sensibilidad) : 0.9056 (211/233)
🎯 PRECISION             : 0

# Riesgo aumentado

In [1]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN GENERAL
# ============================================================

DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_multinivel_filtrado_estado/"

# Rutas de TuckER 5D Balanceado
TUCKER_DIR_FMT = (
    r"C:\Users\56946\TuckER\results"
    r"\Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020patience400_balanceado_5d"
    r"\best_model.pt"
)

# Rutas de Predictores 5D
PRED_DIR_FMT = (
    r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start_multirelacional\predictores_multirelaciona_5d_standard"
    r"\best_predictor_dim5_rdim{rdim}_standard_5d.pt"
)

# Datos
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_20211  = os.path.join(BASE_PATH, "df_20211.csv")
CSV_20212  = os.path.join(BASE_PATH, "df_20212.csv")
# Archivo de Puntajes
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

# Relaciones a rankear
RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]

RDIMS = range(1, 17)
DEVICE = "cpu"

# ============================================================
# 🔹 UTILIDADES
# ============================================================

sys.path.append(r"C:\Users\56946\TuckER")
from load_data import Data

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
    return ckpt

def load_predictor(predictor_path, input_size, out_dim, device="cpu"):
    model = EmbeddingPredictor(input_size, out_dim)
    if not os.path.exists(predictor_path):
        raise FileNotFoundError(f"No existe: {predictor_path}")
    state = torch.load(predictor_path, map_location=device)
    model.load_state_dict(pick_state_dict(state))
    model.to(device).eval()
    return model

def load_tucker_weights(tucker_path, device="cpu"):
    if not os.path.exists(tucker_path):
        raise FileNotFoundError(f"No existe: {tucker_path}")
    raw = torch.load(tucker_path, map_location=device)
    state = pick_state_dict(raw)
    return state["E.weight"].to(device), state["R.weight"].to(device), state["W"].to(device)

def build_vocab(data_dir, reverse=True):
    d = Data(data_dir=data_dir, reverse=reverse)
    entities, relations = d.entities, d.relations
    ent2idx = {e:i for i,e in enumerate(entities)}
    rel2idx = {r:i for i,r in enumerate(relations)}
    return SimpleNamespace(entities=entities, relations=relations,
                           entity_idxs=ent2idx, relation_idxs=rel2idx)

def find_relation(relations, rel2idx, hint="aprueba"):
    cands = [r for r in relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return rel2idx[cands[0]]

@torch.no_grad()
def tucker_score_single(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    elif W.shape[1] == r.numel() and W.shape[0] == d1:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    else: return -9999.0 
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ⚠️ VECTOR 5D (Notas + Puntaje)
def get_notes_vector_5d(csv_path, puntajes_map, max_score, alumno_id, cursos_primer):
    try:
        df = pd.read_csv(csv_path, sep=';')
    except: return torch.zeros((1, 5), dtype=torch.float32) 
    
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    df['ID'] = df['ID'].astype(str).str.strip().str.upper()
    
    subset = df[(df['ID'] == alumno_id) & (df['CURSO'].isin(cursos_primer))].copy()
    subset['NOTA'] = pd.to_numeric(subset['NOTA'], errors='coerce').fillna(0.0)
    
    vec_notas = np.zeros(4, dtype=np.float32)
    idx_map = {c: i for i, c in enumerate(cursos_primer)}
    
    for _, row in subset.iterrows():
        c = row["CURSO"]
        if c in idx_map:
            vec_notas[idx_map[c]] = float(row["NOTA"]) / 7.0
            
    # Puntaje
    score_val = puntajes_map.get(alumno_id, np.nan)
    if pd.isna(score_val): puntaje_norm = -1.0
    else: puntaje_norm = score_val / max_score
        
    final_vec = np.append(vec_notas, puntaje_norm)
    return torch.tensor(final_vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 CARGA Y PREPARACIÓN DE DATOS
# ============================================================

print("Cargando Dataframes...")
df_20211 = pd.read_csv(CSV_20211, sep=';')
df_20212 = pd.read_csv(CSV_20212, sep=';')

# Cargar Puntajes
print("Cargando Puntajes...")
if os.path.exists(RUTA_PUNTAJES):
    df_puntajes = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    max_score_global = df_puntajes["PUNTAJE_PONDERADO"].max()
    if pd.isna(max_score_global): max_score_global = 850.0
    puntajes_map = df_puntajes.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
else:
    print("⚠️ No se encontró archivo de puntajes. Usando -1 por defecto.")
    puntajes_map = {}
    max_score_global = 850.0

for df in (df_20211, df_20212):
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

alumnos = df_20211.groupby("ID")["CURSO"].apply(set)
alumnos_validos = alumnos[alumnos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index.tolist()
print(f"🔹 {len(alumnos_validos)} alumnos válidos 20211.")

df_eval = df_20212[
    (df_20212['ID'].isin(alumnos_validos)) &
    (df_20212['CURSO'].isin(CURSOS_PRED_ALL))
].copy()

def categoria_relacion(nota):
    if pd.isna(nota) or nota < 4.0: return "reprueba"
    elif nota < 5.0: return "aprueba_4_5"
    elif nota < 6.0: return "aprueba_5_6"
    else: return "aprueba_6_7"

df_eval["RELACION_REAL"] = df_eval["NOTA"].apply(categoria_relacion)
print(f"Evaluaciones totales: {len(df_eval)}")

# ============================================================
# 🔹 LOOP EVALUACIÓN
# ============================================================

for rdim in RDIMS:
    print(f"\n============================== Evaluando rdim={rdim} (5D - Riesgo Aumentado) ==============================")

    tucker_path = TUCKER_DIR_FMT.format(rdim=rdim)
    predictor_path = PRED_DIR_FMT.format(rdim=rdim)

    if not os.path.exists(tucker_path):
        print(f"⚠️ Faltan archivos TuckER")
        continue

    d = build_vocab(DATA_DIR, reverse=True)
    E, R, W = load_tucker_weights(tucker_path, DEVICE)
    d1 = E.shape[1]

    rel2idx_map = {}
    for rel_hint in RELACIONES:
        idx = find_relation(d.relations, d.relation_idxs, hint=rel_hint)
        if idx is not None: rel2idx_map[rel_hint] = idx

    try:
        predictor = load_predictor(predictor_path, input_size=5, out_dim=d1, device="cpu")
    except Exception as e:
        print(f"⚠️ Error cargando predictor: {e}")
        continue

    ehat_by_alumno = {}
    for aid in df_eval["ID"].unique():
        x = get_notes_vector_5d(CSV_20211, puntajes_map, max_score_global, aid, CURSOS_PRIMER)
        with torch.no_grad(): ehat_by_alumno[aid] = predictor(x).squeeze(0)

    # 1. Binarización Riesgo Aumentado (Reprueba + Aprueba 4-5 = Riesgo)
    y_true_bin = []
    y_pred_bin = []

    # Definir qué relaciones cuentan como Riesgo
    RIESGO_RELS = ["reprueba", "aprueba_4_5"]

    for _, row in df_eval.iterrows():
        aid, curso, true_rel = row["ID"], row["CURSO"], row["RELACION_REAL"]
        if curso not in d.entity_idxs: continue
        
        t_idx = d.entity_idxs[curso]
        e_hat = ehat_by_alumno[aid]

        scores = {}
        for r_name, r_idx in rel2idx_map.items():
            scores[r_name] = tucker_score_single(e_hat, R, W, E, r_idx, t_idx)
        
        pred_rel = max(scores, key=scores.get)
        
        # --- Lógica de Riesgo Aumentado ---
        # Si la relación real es "reprueba" o "aprueba_4_5", es Riesgo (1)
        # Si la relación predicha es "reprueba" o "aprueba_4_5", predice Riesgo (1)
        
        real_bin = 1 if any(risk in true_rel.lower() for risk in RIESGO_RELS) else 0
        pred_bin = 1 if any(risk in pred_rel.lower() for risk in RIESGO_RELS) else 0
        
        y_true_bin.append(real_bin)
        y_pred_bin.append(pred_bin)

    if not y_true_bin: continue

    # Métricas Binarias
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin).ravel()
    
    total_reales_riesgo = tp + fn
    total_predichos_riesgo = tp + fp
    
    recall = tp / total_reales_riesgo if total_reales_riesgo > 0 else 0
    precision = tp / total_predichos_riesgo if total_predichos_riesgo > 0 else 0
    accuracy = (tp + tn) / len(y_true_bin)
    
    print(f"📊 Resultados Riesgo Aumentado (Repro + Nota 4-5)")
    print("-" * 60)
    print(f"Accuracy Global          : {accuracy:.4f}")
    print(f"Total REALES en Riesgo   : {total_reales_riesgo} (Repro + Nota 4-5)")
    print("-" * 60)
    print(f"✅ RECALL (Sensibilidad) : {recall:.4f}")
    print(f"   (Detectamos {tp} de {total_reales_riesgo} casos)")
    print("-" * 60)
    print(f"🎯 PRECISION             : {precision:.4f}")
    print(f"Matriz: TP={tp}, FN={fn}, FP={fp}, TN={tn}")

Cargando Dataframes...
Cargando Puntajes...
🔹 830 alumnos válidos 20211.
Evaluaciones totales: 3023

============================== Evaluando rdim=1 (5D - Riesgo Aumentado) ==============================
📊 Resultados Riesgo Aumentado (Repro + Nota 4-5)
------------------------------------------------------------
Accuracy Global          : 0.7327
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.0000
   (Detectamos 0 de 808 casos)
------------------------------------------------------------
🎯 PRECISION             : 0.0000
Matriz: TP=0, FN=808, FP=0, TN=2215

============================== Evaluando rdim=2 (5D - Riesgo Aumentado) ==============================
📊 Resultados Riesgo Aumentado (Repro + Nota 4-5)
------------------------------------------------------------
Accuracy Global          : 0.2673
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
-------------------------------------------------

📊 Resultados Riesgo Aumentado (Repro + Nota 4-5)
------------------------------------------------------------
Accuracy Global          : 0.2673
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 1.0000
   (Detectamos 808 de 808 casos)
------------------------------------------------------------
🎯 PRECISION             : 0.2673
Matriz: TP=808, FN=0, FP=2215, TN=0
